<a href="https://colab.research.google.com/github/Farida-AI-FutureProof/5m-data-2.5-data-warehouse/blob/main/assignment_2_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q google-cloud-bigquery

import pandas as pd
from google.cloud import bigquery
from google.colab import auth


In [9]:
from google.colab import files
uploaded = files.upload()


Saving austin_bikeshare_trips.csv to austin_bikeshare_trips.csv
Saving austin_bikeshare_stations.csv to austin_bikeshare_stations.csv


In [10]:
import pandas as pd

# 1. Load the file you just uploaded
df = pd.read_csv('austin_bikeshare_trips.csv')

# 2. Quick sanity check on columns
print(df.columns.tolist())
df.head()


['bikeid', 'checkout_time', 'duration_minutes', 'end_station_id', 'end_station_name', 'month', 'start_station_id', 'start_station_name', 'start_time', 'subscriber_type', 'trip_id', 'year']


,bikeid,checkout_time,duration_minutes,end_station_id,end_station_name,month,start_station_id,start_station_name,start_time,subscriber_type,trip_id,year
0,8.0,19:12:00,41,2565.0,Trinity & 6th Street,3.0,2536.0,Waller & 6th St.,2015-03-19 19:12:00,Walk Up,9900082882,2015.0
1,141.0,2:06:04,6,2570.0,South Congress & Academy,10.0,2494.0,2nd & Congress,2016-10-30 02:06:04,Local365,12617682,2016.0
2,578.0,16:28:27,13,2498.0,Convention Center / 4th St. @ MetroRail,3.0,2538.0,Bullock Museum @ Congress & MLK,2016-03-11 16:28:27,Local365,9075366,2016.0
3,555.0,15:12:00,80,2712.0,Toomey Rd @ South Lamar,11.0,2497.0,Capitol Station / Congress & 11th,2014-11-23 15:12:00,24-Hour Kiosk (Austin B-cycle),9900319298,2014.0
4,86.0,15:39:13,25,3377.0,MoPac Pedestrian Bridge @ Veterans Drive,4.0,2707.0,Rainey St @ Cummings,2017-04-16 15:39:13,Walk Up,14468597,2017.0


In [11]:
# 3. Normalise column names a bit
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

START_COL = 'start_station_name'
END_COL   = 'end_station_name'
DUR_COL   = 'duration_minutes'   # change if your column is named differently

for col in [START_COL, END_COL, DUR_COL]:
    if col not in df.columns:
        raise ValueError(f"Column {col} not found; actual cols = {df.columns.tolist()}")

# 4. Create duration in seconds (to mirror your dbt logic)
df['duration_seconds'] = df[DUR_COL] * 60

# 5. STARTS table
starts = (
    df[df[START_COL].notna()]
      .groupby(START_COL, as_index=False)
      .agg(
          total_duration_starts=('duration_seconds', 'sum'),
          total_starts=(START_COL, 'size')
      )
      .rename(columns={START_COL: 'station_name'})
)

# 6. ENDS table
ends = (
    df[df[END_COL].notna()]
      .groupby(END_COL, as_index=False)
      .agg(
          total_duration_ends=('duration_seconds', 'sum'),
          total_ends=(END_COL, 'size')
      )
      .rename(columns={END_COL: 'station_name'})
)

# 7. Master station list (union)
stations = pd.DataFrame({
    'station_name': pd.concat([starts['station_name'], ends['station_name']]).dropna().unique()
})

# 8. Final dim_station equivalent
dim_station = (
    stations
    .merge(starts, on='station_name', how='left')
    .merge(ends,   on='station_name', how='left')
)

for col in ['total_duration_starts', 'total_duration_ends', 'total_starts', 'total_ends']:
    dim_station[col] = dim_station[col].fillna(0)

dim_station['total_duration'] = (
    dim_station['total_duration_starts'] + dim_station['total_duration_ends']
)

dim_station = dim_station[
    ['station_name', 'total_duration', 'total_starts', 'total_ends']
].sort_values('station_name').reset_index(drop=True)

dim_station.head(20)


,station_name,total_duration,total_starts,total_ends
0,11th & San Jacinto,8139180.0,2985.0,1994
1,13th & San Antonio,1973940.0,527.0,543
2,17th & Guadalupe,22484340.0,7423.0,5631
3,2nd & Congress,99849840.0,26612.0,29516
4,3rd & West,28528020.0,15576.0,13363
5,4th & Congress,71378940.0,24972.0,27902
6,5th & Bowie,60415860.0,26669.0,25070
7,5th & San Marcos,18235260.0,7281.0,7101
8,6th & Congress,8829120.0,2397.0,2601
9,6th & Navasota St.,5202180.0,1301.0,1032
